# rot3 WT-MetaD on an A100

Shuttle pipeline (same as rot1 / rot2htpuma), **not** the dethreading pipeline.
`rot3.txt` carries the rod/wheel SMILES; `rot3_md_seed.xyz` (128 atoms) the coords. Cell 5 checks they agree before building.

**Runtime → Change runtime type → A100.** Check the GPU in cell 1 before spending time.

C52 F6 H54 N2 O14. With a plain 24-crown-8 wheel (`O1CCOCCOCCOCCOCCOCCOCCOCC1`,
the SMILES atom count equals the xyz atom count, so a wrong formula fails fast
rather than silently producing garbage.

`make_plumed.py` additionally asserts the rod has exactly 2 amide N and the wheel
exactly 8 crown O — consistent with N2 / O14 above.

### Budget
~10 h of A100. Cell 8 sets `STEPS`. An A100 should beat the ~0.54 ms/step an A10G
gave on a 21k-atom dethread system; cell 7 measures the real rate on *this* system
and prints what STEPS costs before you commit.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
# Expect A100. If this says T4/V100, change the runtime type before continuing.

In [ ]:
# Conda is required: openmm-plumed has no pip wheel.
!pip install -q condacolab
import condacolab; condacolab.install()
# NOTE: this restarts the kernel. Re-run from the NEXT cell afterwards.

In [ ]:
!mamba install -q -y -c conda-forge openmm openmm-plumed mdtraj \
    openff-toolkit openmmforcefields rdkit
import openmm, openmm.app  # noqa
print('openmm', openmm.version.version)
print('platforms', [openmm.Platform.getPlatform(i).getName()
                    for i in range(openmm.Platform.getNumPlatforms())])

In [ ]:
REPO = 'https://github.com/MauricioCafiero/MD_openmm.git'   # <-- your remote
BRANCH = 'main'
!git clone -q --branch $BRANCH --depth 1 $REPO /content/MD_openmm || echo 'clone failed'
%cd /content/MD_openmm
!pip install -q -e .          # provides the `omd` console script
!ls -l rot3_md_seed.xyz

In [ ]:
# rot3 SMILES come from rot3.txt in the repo (dibenzo-24-crown-8
# wheel, asymmetric rod: one CF3-aryl amide stopper, one dimethyl-isophthalate end).
!cat rot3.txt

# formula check against the seed before building anything
from rdkit import Chem
from collections import Counter
t=dict(l.split(': ',1) for l in open('rot3.txt').read().strip().splitlines())
m=Chem.AddHs(Chem.MolFromSmiles(t['rod']+'.'+t['wheel']))
smi=Counter(a.GetSymbol() for a in m.GetAtoms())
xyz=Counter(l.split()[0] for l in open('rot3_md_seed.xyz').read().splitlines()[2:] if l.strip())
print('SMILES:',dict(smi)); print('seed  :',dict(xyz))
assert smi==xyz, 'formula mismatch -- build_rotaxane.py asserts on this too'
print('OK')


In [ ]:
# Build topology + solvated system (README steps 0-1)
%cd /content/MD_openmm/rotaxanes/metad
!python ../build_rotaxane.py --smiles ../../rot3.txt --from-xyz ../../rot3_md_seed.xyz \
    --out ../outputs_rot3/complex.sdf
!omd build-multimol --sdf ../outputs_rot3/complex.sdf --out-dir ../outputs_rot3
!python make_plumed.py --topology ../outputs_rot3/complex.pdb --out plumed_rot3.dat
!grep -E 'ATOMS|SIGMA|HEIGHT|PACE|BIASFACTOR|GRID' plumed_rot3.dat

In [ ]:
# Measure the ACTUAL rate on this GPU before committing hours.
# Warmup is discarded: OpenMM's CUDA platform compiles/autotunes lazily, and an
# un-warmed probe reads ~2-5x too slow. (Learned the expensive way on A10G.)
import time, openmm as mm
from openmm import app, unit, XmlSerializer
from openmmplumed import PlumedForce

pdb = app.PDBFile('../outputs_rot3/complex.pdb')
system = XmlSerializer.deserializeSystem(open('../outputs_rot3/system.xml').read())
system.addForce(mm.MonteCarloBarostat(1*unit.atmosphere, 300*unit.kelvin, 25))
system.addForce(PlumedForce(open('plumed_rot3.dat').read()))
integ = mm.LangevinMiddleIntegrator(300*unit.kelvin, 1/unit.picosecond, 2*unit.femtosecond)
integ.setConstraintTolerance(1e-6)
sim = app.Simulation(pdb.topology, system, integ, mm.Platform.getPlatformByName('CUDA'))
sim.context.setPositions(pdb.positions); sim.context.applyConstraints(1e-6)
sim.minimizeEnergy(maxIterations=5000)
sim.context.setVelocitiesToTemperature(300*unit.kelvin)
sim.step(2000)                                   # warmup, untimed
t0 = time.time(); sim.step(5000); ms = (time.time()-t0)/5000*1000
print(f'{pdb.topology.getNumAtoms()} atoms -> {ms:.3f} ms/step')
for h in (2, 5, 8, 10):
    print(f'  {h:2d} h A100 = {h*3600/(ms/1000)/1e6:6.2f} M steps = {h*3600/(ms/1000)*2/1e6:5.2f} ns')

In [ ]:
# Production. Set STEPS from cell 7's table, leaving margin -- Colab will cut
# you off mid-run without warning and only completed writes survive.
STEPS = 8_000_000        # <-- set me
OUT   = '/content/drive/MyDrive/rot3_metad'   # Drive so a disconnect is survivable

from google.colab import drive; drive.mount('/content/drive')
import os; os.makedirs(OUT, exist_ok=True)
!python run_metad.py --system ../outputs_rot3/system.xml \
    --topology ../outputs_rot3/complex.pdb \
    --plumed plumed_rot3.dat --out-dir $OUT \
    --steps $STEPS --platform CUDA

In [ ]:
# Check-in: FES from HILLS (safe mid-flight; reads only)
!plumed sum_hills --hills $OUT/HILLS --mintozero --outfile /content/fes.dat
import numpy as np
f=np.loadtxt('/content/fes.dat',comments='#'); k=(f[:,1]-f[:,1].min())/4.184
print(f'cv {f[k.argmin(),0]:+.2f} A is the minimum; max on grid {k.max():.2f} kcal/mol')
